# Syrus 02 — ledger revision review

Review of the revised findings ledger and candidate model against the frozen detector evidence. The decisions are recorded in `docs/syrus/LEDGER_REVIEW.md`; `syrus_01_case_evidence_review.ipynb` reviewed the first ledger version and is left as it was.

**Questions.**
1. Does the fairer smoothing-memory criterion (alert-bin raw score below the previous smoothed score) select different cases than the earlier raw-score-below-threshold test?
2. What does the persistence rule imply about which bins before each alert were threshold hits?
3. How far does a single-group repair differ from removing that group's own contribution, and how much of the alert-bin score can the unedited smoothing state from before the repair window hold?
4. Which evidence-timing facts hold without cut-offs, and how robust is the 1% display floor?
5. Do the candidate and test tables claim anything the evidence does not support?

**Inputs (read-only).** `docs/syrus/cases/*.json`, built by `python -m scripts.build_syrus_cases` from the frozen reports, and `docs/xai_verification_report.json` for the shares of groups below the display floor.

**Outputs.** Inline tables, also saved as CSV to `outputs/syrus_02/` (gitignored). Nothing under `docs/` is written, no frozen notebook is executed, and no detector value is recomputed: every number is a frozen value or arithmetic on frozen values.

**Reproduce.** From the repository root: `python -m scripts.build_syrus_cases --check`, then run this notebook top to bottom.

In [1]:
import json
import sys
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().resolve()
if not (ROOT / "backend").exists() and (ROOT.parent / "backend").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from backend.syrus_app.records import Sources, check_case, hhmm
from scripts.build_syrus_cases import OUT, build_files

OUT_DIR = ROOT / "outputs" / "syrus_02"
OUT_DIR.mkdir(parents=True, exist_ok=True)
pd.set_option("display.max_colwidth", 220)
pd.set_option("display.width", 220)

fresh = build_files(Sources(ROOT))
on_disk = {p.name: json.loads(p.read_text("utf-8")) for p in OUT.glob("*.json")}
assert on_disk == fresh, "case files are stale: run python -m scripts.build_syrus_cases"
for name, case in on_disk.items():
    if name != "manifest.json":
        check_case(case)

manifest = on_disk["manifest.json"]
retro = [on_disk[c["files"]["retro"]["path"]] for c in manifest["cases"]]
R = {c["case_id"]: {r["id"]: r for r in c["records"]} for c in retro}
F = {c["case_id"]: {f["id"]: f for f in c["findings"]} for c in retro}
label = {c["case_id"]: c["alert_time"][:16] for c in retro}
featured = next(c["case_id"] for c in retro if c["role"] == "featured")


def save(df, name):
    df.to_csv(OUT_DIR / f"{name}.csv", index=False)
    return df


print("schema", retro[0]["schema"], "| manifest", manifest["sha256"][:12], "| alerts:", list(label.values()))

schema syrus.case.v2 | manifest c13a63d4ebd2 | alerts: ['2020-04-18 00:40', '2020-05-29 21:20', '2020-06-05 10:10', '2020-07-15 08:45']


## 1. Revised ledger across the four cases

`fires` says whether a rule's condition holds. Asset-wide findings (identifiability, bridge) and benchmark context fire for every case. The featured alert's texts follow.

In [2]:
ledger = pd.DataFrame({label[c]: {f: F[c][f]["fires"] for f in F[c]} for c in F})
ledger["visibility"] = [F[featured][f]["visibility"] for f in ledger.index]
save(ledger.reset_index(names="finding"), "ledger_fires")

,finding,2020-04-18 00:40,2020-05-29 21:20,2020-06-05 10:10,2020-07-15 08:45,visibility
0,F-MARGIN,True,True,True,True,both
1,F-EWMA-MEMORY,False,True,False,True,both
2,F-EVIDENCE-TIMING,True,True,True,True,both
3,F-CF-NONUNIQUE,False,True,False,True,both
4,F-CF-PERSIST,False,False,False,True,both
5,F-TRAIN-RANGE,True,True,True,True,both
6,F-GAP,True,False,False,False,both
7,F-CANDIDATES,True,True,True,True,both
8,F-IDENT,True,True,True,True,both
9,F-BRIDGE,True,True,True,True,both


In [3]:
for f in F[featured].values():
    print(f"[{f['id']}] {f['text']}")
    if f["detail"]:
        print(f"    detail: {f['detail']}")

[F-MARGIN] The smoothed score was 58,989 against a threshold of 52,942 (+11.4%).
[F-EWMA-MEMORY] The alert bin's own raw score (35.8) was below the smoothed score carried from 21:15 (73,727), so the smoothed score was falling at the alert: 99.99% of it came from earlier bins through smoothing.
    detail: Smoothing rule: smoothed = 0.20 × raw + 0.80 × previous smoothed; restarts after a gap over 10 min. The share from earlier bins is one minus the alert bin's own term. This shows how the smoothing works; it does not show which earlier bins carried the evidence or that a condition persisted, and one large raw score can by itself produce several smoothed hits.
[F-EVIDENCE-TIMING] Calibration percentile of each group's smoothed contribution: dv_pressure 69.82 at 20:25 and 99.55 at 21:20, largest one-bin rise 35.08 to 100 between 21:05 and 21:10, first above every calibration bin at 21:10; tp2 99.94 at 20:25 and 99.94 at 21:20, largest one-bin rise 87.31 to 99.78 between 21:05 and 21:10, f

## 2. Smoothing memory: which criterion?

The threshold is the 99.5th percentile of **smoothed** calibration scores, and no single bin can alert under the 3-of-4 persistence rule, so comparing a raw score with the threshold mixes units. The smoothed update is `s_t = 0.2 x_t + 0.8 s_(t-1)`; the smoothed score falls exactly when `x_t < s_(t-1)`. The share of `s_t` carried from earlier bins is `0.8 s_(t-1) / s_t`; 80% means the score is flat.

In [4]:
rows = []
for c in R:
    r = R[c]
    raw, prev, s, thr = r["alert.raw"]["value"], r["alert.prev_smoothed"]["value"]["score"], r["alert.smoothed"]["value"], r["alert.threshold"]["value"]
    rows.append({"alert": label[c], "raw": round(raw, 1), "previous smoothed": round(prev), "smoothed": round(s), "threshold": round(thr),
                 "raw < threshold (earlier test)": raw < thr, "raw < previous smoothed (revised)": raw < prev,
                 "share from earlier bins": r["alert.memory"]["display"], "finding fires": F[c]["F-EWMA-MEMORY"]["fires"]})
memory = save(pd.DataFrame(rows), "smoothing_memory")
assert (memory["raw < threshold (earlier test)"] == memory["raw < previous smoothed (revised)"]).all()
memory

,alert,raw,previous smoothed,smoothed,threshold,raw < threshold (earlier test),raw < previous smoothed (revised),share from earlier bins,finding fires
0,2020-04-18 00:40,197231.2,176258,180453,52942,False,False,78.14%,False
1,2020-05-29 21:20,35.8,73727,58989,52942,True,True,99.99%,True
2,2020-06-05 10:10,154559.4,81616,96205,52942,False,False,67.87%,False
3,2020-07-15 08:45,22696.3,73697,63497,52942,True,True,92.85%,True


## 3. What the persistence rule implies about recent hits

Each explained bin is the start of its alert: notebook 04 explains the first timely alert episode by its start time, and `extract_alert_episodes` starts an episode at the first alerting bin and merges alerting bins up to 30 minutes apart. So no bin in the 30 minutes before (since the last restart) alerted. Together with the known hits (the alert bin, and the previous bin through the back-calculated smoothed score), the builder enumerates every hit pattern over the bins since the last restart and keeps only what holds in all of them.

The rule counts **smoothed** hits. One large raw score can produce several: starting from a zero state, a raw score `x` gives smoothed values `0.2x`, `0.16x`, `0.128x`, so `x` at least `1 / 0.128` times the threshold yields three hits on its own.

In [5]:
alpha = R[featured]["detector.ewma"]["value"]["alpha"]
print(f"single-bin bound for three smoothed hits from a zero state: {1 / (alpha * (1 - alpha) ** 2):.4f} x threshold")
hits = pd.DataFrame([{"alert": label[c], "note": R[c]["alert.hits"]["display"],
                      "hits": [hhmm(b) for b in R[c]["alert.hits"]["value"]["hits"]],
                      "exactly one of": [hhmm(b) for b in R[c]["alert.hits"]["value"]["undetermined"]],
                      "misses": [hhmm(b) for b in R[c]["alert.hits"]["value"]["misses"]]} for c in R])
save(hits, "hit_notes")

single-bin bound for three smoothed hits from a zero state: 7.8125 x threshold


,alert,note,hits,exactly one of,misses
0,2020-04-18 00:40,"00:30, 00:35 and 00:40 were at or above the threshold. Persistence history restarted at 00:30 after a gap.","[00:30, 00:35, 00:40]",[],[]
1,2020-05-29 21:20,21:15 and 21:20 were at or above the threshold; exactly one of 21:05 and 21:10 was; 21:00 was below it.,"[21:15, 21:20]","[21:05, 21:10]",[21:00]
2,2020-06-05 10:10,10:05 and 10:10 were at or above the threshold; exactly one of 09:55 and 10:00 was; 09:50 was below it.,"[10:05, 10:10]","[09:55, 10:00]",[09:50]
3,2020-07-15 08:45,08:40 and 08:45 were at or above the threshold; exactly one of 08:30 and 08:35 was; 08:25 was below it.,"[08:40, 08:45]","[08:30, 08:35]",[08:25]


## 4. Repairs: path, reduction ratio and the pre-window bound

A repair replaces one group's robust-scaled coordinates with the PCA reconstruction over the window, re-projects and rescores each bin, and reruns smoothing and persistence over the TEST series. Re-projection changes every group's residual, so the score reduction need not equal the group's own contribution. `naive` subtracts only the group's own smoothed contribution from the alert-bin smoothed score, for comparison.

In [6]:
rows = []
for c in R:
    r = R[c]
    s, thr = r["alert.smoothed"]["value"], r["alert.threshold"]["value"]
    for i in [k for k in r if k.startswith("cf.") and k != "cf.prewindow"]:
        v = r[i]["value"]
        rows.append({"alert": label[c], "group": v["group"], "share": round(v["share"], 4), "own contribution": round(v["own_contribution"]),
                     "repaired smoothed": round(v["repaired_smoothed_score"]), "cleared": v["alert_cleared"], "path": v["path"],
                     "reduction / own": round(v["reduction_ratio"], 3) if v["own_contribution"] >= 1 else None,
                     "naive": round(s - v["own_contribution"]), "naive below threshold": s - v["own_contribution"] < thr})
repairs = save(pd.DataFrame(rows), "repairs")
repairs[repairs["share"] >= 0.01]

,alert,group,share,own contribution,repaired smoothed,cleared,path,reduction / own,naive,naive below threshold
0,2020-04-18 00:40,tp2,0.9397,169575,10874,True,score,1.000,10878,True
1,2020-04-18 00:40,dv_pressure,0.0602,10862,169570,False,NaN,1.002,169592,False
5,2020-05-29 21:20,dv_pressure,0.7756,45754,7472,True,score,1.126,13235,True
6,2020-05-29 21:20,tp2,0.1263,7453,51473,True,score,1.009,51536,True
7,2020-05-29 21:20,motor_current,0.0980,5782,48374,True,score,1.836,53207,False
10,2020-06-05 10:10,tp2,0.8607,82806,13460,True,score,0.999,13399,True
11,2020-06-05 10:10,dv_pressure,0.1302,12524,82955,False,NaN,1.058,83681,False
15,2020-07-15 08:45,dv_pressure,0.7450,47303,9929,True,score,1.132,16194,True
16,2020-07-15 08:45,tp2,0.1380,8760,54996,True,persistence,0.970,54737,False
17,2020-07-15 08:45,motor_current,0.1170,7432,49993,True,score,1.817,56064,False


In [7]:
mc = repairs[repairs["group"] == "motor_current"][["alert", "share", "reduction / own", "naive", "naive below threshold", "cleared"]]
print("motor_current in every case: the actual repair removes about 1.8x its own contribution")
mc

motor_current in every case: the actual repair removes about 1.8x its own contribution


,alert,share,reduction / own,naive,naive below threshold,cleared
2,2020-04-18 00:40,0.0001,1.838,180444,False,False
7,2020-05-29 21:20,0.0980,1.836,53207,False,True
12,2020-06-05 10:10,0.0090,1.795,95335,False,False
17,2020-07-15 08:45,0.1170,1.817,56064,False,True


In [8]:
rows = []
for c in R:
    r = R[c]
    pre, w = r["cf.prewindow"]["value"], r["temporal.window"]["value"]
    rows.append({"alert": label[c], "window bins": w["observed"], "0.8 ** bins (coefficient only)": round(0.8 ** w["observed"], 4),
                 "restart in window": pre["restart"] and hhmm(pre["restart"]), "bound on pre-window share": r["cf.prewindow"]["display"]})
save(pd.DataFrame(rows), "prewindow_bound")

,alert,window bins,0.8 ** bins (coefficient only),restart in window,bound on pre-window share
0,2020-04-18 00:40,10,0.1074,00:30,"none, because smoothing restarts at 00:30 inside the window"
1,2020-05-29 21:20,12,0.0687,NaN,at most 12.7%
2,2020-06-05 10:10,12,0.0687,NaN,at most 14.0%
3,2020-07-15 08:45,12,0.0687,NaN,at most 15.6%


The coefficient on the smoothing state before the window is `0.8^bins` (about 6.9% after 12 uninterrupted bins), but its share of the alert-bin score depends on how large that state was, which is not exported. Because every repaired raw score is non-negative and the earlier state is not edited, that state's contribution is at most the lowest repaired smoothed score; after a restart inside the window it is zero.

## 5. Evidence timing without cut-offs, and the display floor

Per displayed group: calibration percentile at the window start and at the alert, the largest one-bin rise, and the first bin whose smoothed contribution is above every calibration bin. No 0.50 or 0.99 cut-offs.

In [9]:
rows = []
for c in R:
    for g, o in F[c]["F-EVIDENCE-TIMING"]["output"].items():
        rise = o["rise"]
        rows.append({"alert": label[c], "group": g, "start": round(100 * o["start"], 2), "alert bin": round(100 * o["end"], 2),
                     "largest rise": rise and f"{hhmm(rise['from'])}->{hhmm(rise['to'])} (+{100 * rise['size']:.2f})" + (" gap" if rise["across_gap"] else ""),
                     "first above all calibration bins": o["first_above_all"] and hhmm(o["first_above_all"])})
save(pd.DataFrame(rows), "evidence_timing")

,alert,group,start,alert bin,largest rise,first above all calibration bins
0,2020-04-18 00:40,tp2,1.34,100.00,00:15->00:30 (+98.66) gap,00:30
1,2020-04-18 00:40,dv_pressure,0.00,90.70,00:15->00:30 (+90.37) gap,NaN
2,2020-05-29 21:20,dv_pressure,69.82,99.55,21:05->21:10 (+64.92),21:10
3,2020-05-29 21:20,tp2,99.94,99.94,21:05->21:10 (+12.47),21:15
4,2020-05-29 21:20,motor_current,47.38,99.05,21:05->21:10 (+64.42),NaN
5,2020-06-05 10:10,tp2,87.53,100.00,09:50->09:55 (+15.03),09:55
6,2020-06-05 10:10,dv_pressure,72.33,92.04,09:50->09:55 (+30.68),NaN
7,2020-07-15 08:45,dv_pressure,99.55,99.55,08:25->08:30 (+6.35),NaN
8,2020-07-15 08:45,tp2,100.00,100.00,08:25->08:30 (+0.06),07:50
9,2020-07-15 08:45,motor_current,99.28,99.55,08:25->08:30 (+5.07),NaN


In [10]:
ver = json.loads((ROOT / "docs" / "xai_verification_report.json").read_text("utf-8"))
floor = Sources(ROOT).load("configs/syrus_cases.yaml")["display_floor"]
hidden = max(g["share"] for i in ver["incidents"] for g in i["ranked_groups"] if g["share"] < floor)
shown = min(g["share"] for i in ver["incidents"] for g in i["ranked_groups"] if g["share"] >= floor)
print(f"display floor {floor:.2%}: any floor above {hidden:.2%} and up to {shown:.2%} shows the same groups in all four cases")

display floor 1.00%: any floor above 0.90% and up to 6.02% shows the same groups in all four cases


## 6. Candidates, tests and pair limits

Every candidate is listed for every case, in catalog order, with status `open`. Tests are drafts with no rule and no result; a pair is limited when every test comparing it needs an external observation.

In [11]:
cands = pd.DataFrame([{"candidate": r["display"]["label"], "layer": r["value"]["layer"], "status": r["value"]["status"],
                       **{label[c]: R[c][r["id"]]["display"]["related"] for c in R}}
                      for r in R[featured].values() if r["kind"] == "hypothesis"])
save(cands, "candidates")

,candidate,layer,status,2020-04-18 00:40,2020-05-29 21:20,2020-06-05 10:10,2020-07-15 08:45
0,Air leakage,process,open,"tp2, dv_pressure","dv_pressure, tp2","tp2, dv_pressure","dv_pressure, tp2"
1,Higher downstream air demand,process,open,"tp2, dv_pressure","dv_pressure, tp2","tp2, dv_pressure","dv_pressure, tp2"
2,Pressure-regulation behaviour,process,open,dv_pressure,dv_pressure,dv_pressure,dv_pressure
3,Compressor performance change,process,open,tp2,"tp2, motor_current",tp2,"tp2, motor_current"
4,Normal operating change,process,open,no specific signal group,no specific signal group,no specific signal group,no specific signal group
5,Sensor or measurement problem,evidence,open,"tp2, dv_pressure","dv_pressure, tp2","tp2, dv_pressure","dv_pressure, tp2"
6,Data-quality effect,evidence,open,no specific signal group,no specific signal group,no specific signal group,no specific signal group
7,Reference-coverage gap,evidence,open,no specific signal group,no specific signal group,no specific signal group,no specific signal group


In [12]:
case0 = retro[0]
tests = pd.DataFrame([{"test": t["id"], "compares": "; ".join(" vs ".join(p) for p in t["compares"]),
                       "availability": ", ".join(o["availability"] for o in t["observations"]), "maturity": t["maturity"], "result": t["result"]}
                      for t in case0["tests"]])
print("pair limits:", [(lim["pair"], lim["text"]["en"]) for lim in case0["pair_limits"]])
save(tests, "tests")

pair limits: [(['air_leakage', 'downstream_demand'], 'cannot be told apart with MetroPT-3 data')]


,test,compares,availability,maturity,result
0,t_demand_record,air_leakage vs downstream_demand,external,draft,not_run
1,t_cycle_pattern,pressure_regulation vs air_leakage; pressure_regulation vs downstream_demand; pressure_regulation vs normal_operating_change,raw_dataset,draft,not_run
2,t_sensor_agreement,sensor_problem vs air_leakage; sensor_problem vs downstream_demand; sensor_problem vs pressure_regulation; sensor_problem vs compressor_performance,raw_dataset,draft,not_run
3,t_load_response,compressor_performance vs air_leakage; compressor_performance vs downstream_demand,raw_dataset,draft,not_run
4,t_raw_persistence,normal_operating_change vs air_leakage; normal_operating_change vs downstream_demand; normal_operating_change vs compressor_performance,raw_dataset,draft,not_run
5,t_bin_coverage,data_quality vs sensor_problem; data_quality vs reference_coverage,raw_dataset,draft,not_run
6,t_reference_occurrence,reference_coverage vs sensor_problem; reference_coverage vs data_quality,raw_dataset,draft,not_run


## Reading

- **Smoothing memory.** The revised criterion selects the same two alerts as the earlier test (2020-05-29 21:20 and 2020-07-15 08:45), now for a reason that compares like with like. At 21:20 the bin's own raw score was 35.8 against 73,727 carried from 21:15, so 99.99% of the smoothed score came from earlier bins. The two strong alerts were still rising (78.14% and 67.87% carried).
- **Recent hits.** For the three alerts with complete windows, the alert bin and the bin before it were hits, exactly one of the two bins before those was, and the bin before that was not. The first alert followed a 15-minute gap, so its three bins since the 00:30 restart were all hits. Which bin was the hit cannot be determined, and a single raw score of at least about 7.81 times the threshold could produce three smoothed hits on its own, so persistence does not show three separate anomalous bins.
- **Repairs.** The `motor_current` repair removes about 1.8 times its own smoothed contribution in all four cases. For the two alerts where it clears, subtracting only its own contribution would leave the score above the threshold (53,207 and 56,064 against 52,942). The `tp2` repair of 2020-07-15 08:45 clears through persistence with the smoothed score still at 54,996. The unedited smoothing state from before the window accounts for at most 12.7%, 14.0% and 15.6% of the three complete-window alerts and none of the first.
- **Evidence timing.** At 21:20, `dv_pressure` first ranked above every calibration bin at 21:10, `motor_current` rose in the same step, and `tp2` was already at the 99.94th percentile at 20:25. At 08:45, `tp2` was above every calibration bin from the window start and no other group ever was.
- **Display floor.** The 1% floor shows the same groups as any floor above 0.90% and up to 6.02%.
- **Candidates.** All eight are open and unranked; the only pair limit is air leakage vs higher downstream demand, whose separating observation is external to MetroPT-3. No test has a rule, so nothing supports or weakens any candidate.
- **Limits.** These are statements about detector evidence. The raw-score series, raw telemetry and units are not loaded, so the size of any change in physical units, and which bin was a hit, cannot be established here.